In [1]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
!cp -n "/content/drive/MyDrive/aai-project/EuroSAT_RGB.zip" /content/
!unzip -q -n /content/EuroSAT_RGB.zip -d /content/eurosat
!ls /content/eurosat/EuroSAT_RGB

Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
AnnualCrop  HerbaceousVegetation  Industrial  PermanentCrop  River
Forest	    Highway		  Pasture     Residential    SeaLake


In [3]:
import os
from google.colab import files
REPO = "/content/Advanced-AI"
os.makedirs(f"{REPO}/src", exist_ok=True)
open(f"{REPO}/src/__init__.py", "a").close()

need = {"transforms.py": f"{REPO}/src/transforms.py", "requirements.txt": f"{REPO}/requirements.txt"}
for name, dest in need.items():
    while not os.path.exists(dest):
        print(f"Choose {name} from your local repo:")
        for got, data in files.upload().items():
            if got.startswith(name.split(".")[0]):
                open(dest, "wb").write(data)
            else:
                print(f"  that was {got}, not {name}")
print("both files in place")
!pip install -q -r {REPO}/requirements.txt

Choose requirements.txt from your local repo:


Saving requirements.txt to requirements.txt
both files in place
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 2.4 MB/s et

In [1]:
   !ls /content/drive/MyDrive/aai-project

EuroSAT_RGB.zip  features


In [2]:
%%time
import sys, hashlib, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch, torchvision, transformers, PIL
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import ViTModel

REPO = "/content/Advanced-AI"
sys.path.insert(0, REPO)
from src.transforms import eight_views       # Eddie's function, so views match LoRA's augmentation

ROOT = Path("/content/eurosat/EuroSAT_RGB")
OUT  = Path("/content/drive/MyDrive/aai-project/features")
CHECKPOINT = "google/vit-base-patch16-224-in21k"
BATCH = 32        # 32 images x 8 views = 256 forward passes; drop to 16 if out of memory

# versions must match requirements.txt
pins = dict(l.strip().split("==") for l in open(f"{REPO}/requirements.txt") if "==" in l)
have = {"torch": torch.__version__, "torchvision": torchvision.__version__,
        "transformers": transformers.__version__, "pillow": PIL.__version__}
for k, v in have.items():
    print(k, v)
    assert v.split("+")[0] == pins[k], f"{k} is {v} but requirements.txt pins {pins[k]}: restart the session after Cell 2"
assert torch.cuda.is_available(), "torch can't see a GPU: check Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("transforms.py sha256:", hashlib.sha256(open(f"{REPO}/src/transforms.py", "rb").read()).hexdigest()[:12])

# same image order as the first extraction: class folders sorted, files sorted
items = [(p.name, p.stem, d.name, str(p)) for d in sorted(ROOT.iterdir()) if d.is_dir() for p in sorted(d.glob("*.jpg"))]
assert len(items) == 27000, len(items)

class EightViews(Dataset):
    def __len__(self): return len(items)
    def __getitem__(self, i): return eight_views(Image.open(items[i][3]).convert("RGB"))   # 8 x 3 x 224 x 224

loader = DataLoader(EightViews(), batch_size=BATCH, shuffle=False, num_workers=2)
model = ViTModel.from_pretrained(CHECKPOINT, add_pooling_layer=False).to("cuda").eval()

feats = np.empty((len(items), 8, 768), dtype=np.float16)
done = 0
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
    for b, x in enumerate(loader):                                   # x: B x 8 x 3 x 224 x 224
        B = x.shape[0]
        cls = model(pixel_values=x.flatten(0, 1).to("cuda")).last_hidden_state[:, 0]   # (B*8) x 768
        feats[done:done + B] = cls.float().view(B, 8, 768).cpu().numpy()
        done += B
        if b % 100 == 0: print(f"{done}/{len(items)} images")
assert done == len(items)

OUT.mkdir(parents=True, exist_ok=True)
np.save("/content/features_8view.npy", feats)
shutil.copy("/content/features_8view.npy", OUT / "features_8view.npy")
index = pd.DataFrame(items, columns=["filename", "stem", "label", "path"]).drop(columns="path")
index.insert(0, "row", range(len(index)))
index.to_csv(OUT / "features_8view_index.csv", index=False)
print("saved", feats.shape, "to", OUT)

torch 2.14.1+cu130
torchvision 0.29.1+cu130
transformers 5.19.0
pillow 12.3.0
GPU: Tesla T4
transforms.py sha256: 05894f3cb64b


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


32/27000 images
3232/27000 images
6432/27000 images
9632/27000 images
12832/27000 images
16032/27000 images
19232/27000 images
22432/27000 images
25632/27000 images
saved (27000, 8, 768) to /content/drive/MyDrive/aai-project/features
CPU times: user 9min 10s, sys: 41.8 s, total: 9min 52s
Wall time: 11min 57s


In [3]:
f = feats.astype(np.float32)
print("any NaN:", bool(np.isnan(f).any()))
unit = f / np.linalg.norm(f, axis=2, keepdims=True)

# 1. view 0 should reproduce yesterday's single-view file
old = np.load(OUT / "fingerprints_view0.npy").astype(np.float32)
old_idx = pd.read_csv(OUT / "fingerprints_index.csv")
assert (old_idx["key"] == index["label"] + "/" + index["stem"]).all(), "row order differs from yesterday's file"
cos0 = (unit[:, 0] * (old / np.linalg.norm(old, axis=1, keepdims=True))).sum(1)
print(f"view 0 vs yesterday: mean cosine {cos0.mean():.5f}, min {cos0.min():.5f}")

# 2. the other views should be close to view 0 but not identical
for v in range(1, 8):
    print(f"view {v} vs view 0: mean cosine {(unit[:, 0] * unit[:, v]).sum(1).mean():.3f}")

any NaN: False
view 0 vs yesterday: mean cosine 1.00000, min 0.99988
view 1 vs view 0: mean cosine 0.786
view 2 vs view 0: mean cosine 0.835
view 3 vs view 0: mean cosine 0.791
view 4 vs view 0: mean cosine 0.965
view 5 vs view 0: mean cosine 0.789
view 6 vs view 0: mean cosine 0.835
view 7 vs view 0: mean cosine 0.794
